In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sovitrath/diabetic-retinopathy-224x224-2019-data")

print("Path to dataset files:", path)

100%|██████████| 238M/238M [00:12<00:00, 20.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/sovitrath/diabetic-retinopathy-224x224-2019-data/versions/4


In [4]:
IMG_SIZE = 224
NORMALIZE = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    NORMALIZE
])

In [6]:
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    NORMALIZE
])

In [7]:
class MapDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.dataset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.dataset)

DATA_DIR = "/root/.cache/kagglehub/datasets/sovitrath/diabetic-retinopathy-224x224-2019-data/versions/4/colored_images"
raw_dataset = datasets.ImageFolder(root=DATA_DIR, transform=None)
class_names = raw_dataset.classes
num_classes = len(class_names)

In [8]:
# Split 80/10/10
train_size = int(0.80 * len(raw_dataset))
val_size = int(0.10 * len(raw_dataset))
test_size = len(raw_dataset) - train_size - val_size

train_subset, val_subset, test_subset = random_split(
    raw_dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(MapDataset(train_subset, train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(MapDataset(val_subset, eval_transform), batch_size=32, shuffle=False)
test_loader = DataLoader(MapDataset(test_subset, eval_transform), batch_size=32, shuffle=False)

# Compute Class Weights for Loss Function
train_targets = [raw_dataset.targets[i] for i in train_subset.indices]
class_weights = compute_class_weight('balanced', classes=np.unique(train_targets), y=train_targets)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float).to(device))

In [9]:
# 2. INITIALIZE PRE-TRAINED EFFICIENTNET-B0 MODEL
weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

# Freeze pre-trained feature extractor backbone
for param in model.features.parameters():
    param.requires_grad = False

# Replace final classification head (1280 features -> 5 DR classes)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, num_classes)
).to(device)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 31.2MB/s]


In [10]:
# 3. STAGE 1: WARMUP CLASSIFIER HEAD ONLY
print("--- STAGE 1: Training Classification Head ---")
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-2)

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            if is_train:
                optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * inputs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

for epoch in range(5):
    t_loss, t_acc = run_epoch(model, train_loader, criterion, optimizer)
    v_loss, v_acc = run_epoch(model, val_loader, criterion)
    print(f"Stage 1 | Epoch {epoch+1}/5 | Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f} Acc: {v_acc:.4f}")

# 4. STAGE 2: FINE-TUNING DEEP BACKBONE LAYERS
print("\n--- STAGE 2: Fine-Tuning Top Convolutional Blocks ---")
# Unfreeze the last 2 MBConv blocks of EfficientNet backbone
for param in model.features[-2:].parameters():
    param.requires_grad = True

# Differential Learning Rates: Lower LR for backbone to preserve pre-trained features
optimizer_ft = optim.AdamW([
    {'params': model.features[-2:].parameters(), 'lr': 1e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4}
], weight_decay=1e-2)

best_val_loss = float('inf')
os.makedirs("checkpoints", exist_ok=True)

for epoch in range(8):
    t_loss, t_acc = run_epoch(model, train_loader, criterion, optimizer_ft)
    v_loss, v_acc = run_epoch(model, val_loader, criterion)
    print(f"Stage 2 | Epoch {epoch+1}/8 | Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f} Acc: {v_acc:.4f}")

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        torch.save(model.state_dict(), "checkpoints/best_dr_transfer_model.pth")
        print(f"   ---> Best model saved with Val Loss: {v_loss:.4f}")

# 5. FINAL TEST EVALUATION
print("\n--- Evaluating Transfer Learning Model on Test Set ---")
model.load_state_dict(torch.load("checkpoints/best_dr_transfer_model.pth"))
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        y_true.extend(labels.numpy())
        y_pred.extend(outputs.argmax(dim=1).cpu().numpy())

print(classification_report(y_true, y_pred, target_names=class_names))


--- STAGE 1: Training Classification Head ---
Stage 1 | Epoch 1/5 | Train Loss: 1.3379 Acc: 0.5685 | Val Loss: 1.1475 Acc: 0.7377
Stage 1 | Epoch 2/5 | Train Loss: 1.1470 Acc: 0.6384 | Val Loss: 1.1029 Acc: 0.7049
Stage 1 | Epoch 3/5 | Train Loss: 1.0729 Acc: 0.6699 | Val Loss: 1.0341 Acc: 0.7213
Stage 1 | Epoch 4/5 | Train Loss: 1.0348 Acc: 0.6746 | Val Loss: 1.0093 Acc: 0.7131
Stage 1 | Epoch 5/5 | Train Loss: 1.0178 Acc: 0.6729 | Val Loss: 0.9712 Acc: 0.7213

--- STAGE 2: Fine-Tuning Top Convolutional Blocks ---
Stage 2 | Epoch 1/8 | Train Loss: 0.9965 Acc: 0.6890 | Val Loss: 0.9668 Acc: 0.7486
   ---> Best model saved with Val Loss: 0.9668
Stage 2 | Epoch 2/8 | Train Loss: 0.9727 Acc: 0.7057 | Val Loss: 0.9656 Acc: 0.7404
   ---> Best model saved with Val Loss: 0.9656
Stage 2 | Epoch 3/8 | Train Loss: 0.9911 Acc: 0.6862 | Val Loss: 0.9562 Acc: 0.7541
   ---> Best model saved with Val Loss: 0.9562
Stage 2 | Epoch 4/8 | Train Loss: 0.9598 Acc: 0.7163 | Val Loss: 0.9500 Acc: 0.7514
  

In [11]:
# --- SAVE ---
torch.save(model.state_dict(), "dr_model_weights.pth")

# --- LOAD ---
# 1. Recreate the model architecture
model = models.efficientnet_b0()
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 5)

# 2. Load the state dict and map to device
model.load_state_dict(torch.load("dr_model_weights.pth", map_location=device))
model.to(device)
model.eval()  # Set to evaluation mode for inference

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat